# 🔵 외부고리 은하 영상 분석 — Google Colab용

이 Notebook은 **조훈 · 손정주의 외부고리 은하 영상 분석 Python 알고리즘**을 Google Colab에서 더 쉽게 실행하도록 정리한 교육용 버전입니다.

### 이 버전에서 달라진 점
- Windows의 `C:\\ring_galaxy` 폴더를 만들 필요가 없습니다.
- 분석할 **은하 번호만 선택하면** `ring_galaxy.csv`와 해당 은하의 SDSS `u/g/r/i/z` FITS 파일 5개를 GitHub에서 자동으로 내려받습니다.
- 원본의 핵심인 **은하 이미지 직접 클릭 방식**을 유지합니다. Colab에서 불안정한 `%matplotlib widget` 대신 브라우저의 JavaScript 클릭 캡처를 사용해 좌표를 정확히 읽습니다.
- 결과 파일은 `/content/ring_galaxy/result/`에 저장됩니다.
- 원본 연구용 Notebook과 분석 흐름은 가능한 범위에서 유지합니다.

> **추천 첫 실행:** 은하 번호 `1010`, 이미지 크기 `300`, smoothing `0.9`로 시작한 뒤 i-band 영상에서 은하의 긴 방향 끝점과 짧은 방향 끝점을 각각 클릭하세요.

관련 논문: **조훈 · 손정주, 「외부고리 은하 영상 분석을 위한 파이썬 기반 알고리즘 개발」, 한국지구과학회지 43(5), 579–590 (2022)**  
DOI: https://doi.org/10.5467/JKESS.2022.43.5.579


> 이 Notebook은 원본 연구용 Jupyter Notebook의 분석 흐름을 축약하지 않고 유지하면서, 로컬 경로·입력·클릭 상호작용만 Google Colab 환경에 맞게 변환한 Colab용 Notebook입니다.


# **데이터 수집 및 탐색**

## **데이터 수집**

### **패키지 라이브러리 불러오기**

* 패키지를 자동으로 설치한 뒤 다음의 알고리즘 실행에 필요한 여러가지 패키지와 라이브러리를 불러옵니다.

In [ ]:
# Colab 실행에 필요한 패키지 설치
!pip -q install astropy regions photutils scikit-learn pandas matplotlib pillow
print("✅ 패키지 설치 완료")


In [ ]:
# 패키지 라이브러리 불러오기
%matplotlib inline

from IPython.display import display, Javascript
from google.colab.output import eval_js

from astropy.wcs import WCS
from astropy.io import fits
from astropy.convolution import Gaussian2DKernel, convolve, convolve_fft
from regions import PixCoord, CirclePixelRegion
from photutils.aperture import EllipticalAperture
from photutils.isophote import EllipseGeometry, Ellipse
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image
from io import BytesIO
import base64
import json
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import numpy as np
import os
import sys

# 원본 연구 Notebook 호환용 click2label helper도 GitHub에서 자동으로 받습니다.
helper_dir = Path("/content/ring_galaxy_helpers")
helper_dir.mkdir(parents=True, exist_ok=True)
helper_path = helper_dir / "click2label.py"
helper_url = "https://raw.githubusercontent.com/GodTANKS/Outer-Ring-Galaxy-DataScience/main/coding/click2label.py"
urlretrieve(helper_url, helper_path)

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import click2label
assert hasattr(click2label, "ClickLabel")
print("✅ 라이브러리 불러오기 완료")
print("✅ click2label.py 불러오기 확인:", click2label.__file__)


### **분석할 데이터 불러오기**

* Colab에서는 별도 폴더를 직접 만들 필요가 없습니다.
* 아래 셀을 실행하면 작업 폴더와 결과 폴더가 자동 생성되고 `ring_galaxy.csv`가 GitHub에서 자동 다운로드됩니다.
* 이후 분석할 은하 번호를 선택하면 그 은하의 `u/g/r/i/z` FITS 파일 5개만 자동 다운로드합니다.


In [ ]:
# Colab 작업 폴더 자동 생성 + 은하 목록 자동 다운로드
BASE_RAW = "https://raw.githubusercontent.com/GodTANKS/Outer-Ring-Galaxy-DataScience/main"

work_dir = Path("/content/ring_galaxy")
result_dir = work_dir / "result"
work_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

often_path = str(work_dir) + "/"
save_path = str(result_dir) + "/"

csv_path = work_dir / "ring_galaxy.csv"
if not csv_path.exists():
    urlretrieve(f"{BASE_RAW}/ring_galaxy.csv", csv_path)

print("✅ 작업 폴더:", work_dir)
print("✅ 결과 저장 폴더:", result_dir)


* GitHub에서 자동으로 내려받은 고리은하 목록을 확인합니다.


In [ ]:
# 고리은하 목록 확인
pd.set_option('display.max_rows', 30)
ring_df = pd.read_csv(often_path + 'ring_galaxy.csv')
display(ring_df)
print(f"총 {len(ring_df)}개 은하가 목록에 있습니다.")


* 아래 `ring_num`에 분석하려는 고리은하 번호를 입력합니다. 처음에는 **1010**을 권장합니다.
* 실행하면 해당 은하의 SDSS `u/g/r/i/z` FITS 파일만 자동으로 다운로드합니다.


In [ ]:
#@title ① 분석할 외부고리 은하 선택
ring_num = 1010 #@param {type:"integer"}

ring = ring_df['object_number'] == ring_num
ring_info = ring_df[ring]
if ring_info.empty:
    raise ValueError(f"{ring_num} 은하는 ring_galaxy.csv 목록에 없습니다.")

display(ring_info)
ring_ra = ring_info['ra'].values[0]
ring_dec = ring_info['dec'].values[0]

for band in ['u','g','r','i','z']:
    filename = f"{ring_num}_{band}.fits"
    target = work_dir / filename
    if not target.exists():
        print("다운로드:", filename)
        urlretrieve(f"{BASE_RAW}/{filename}", target)

print("✅ 선택한 은하의 FITS 5개 다운로드 완료")


In [ ]:
# 각 필터 FITS 파일 불러오기
u = fits.open(often_path + str(ring_num)+'_u.fits')
g = fits.open(often_path + str(ring_num)+'_g.fits')
r = fits.open(often_path + str(ring_num)+'_r.fits')
i = fits.open(often_path + str(ring_num)+'_i.fits')
z = fits.open(often_path + str(ring_num)+'_z.fits')

filter = ['u', 'g', 'r', 'i', 'z']
print("✅ FITS 파일 로드 완료:", filter)


## **데이터 탐색(EDA)**

### **데이터 기본 통계 정보**

* 분석하려는 은하의 fits 파일의 헤더 정보와 데이터 구조, 기본 통계 정보를 확인합니다.

In [ ]:
# 각 필터 fits 파일 정보

for f in filter:
    print(globals()['{}'.format(f) ].info())
    print('------------------------------------------------')

In [ ]:
# 각 필터 fits 파일 헤더, 데이터 상세 정보

for f in filter:
    globals()['hdr_{}'.format(f) ] = globals()[ '{}'.format(f) ][0].header # 헤더 정보
    globals()['dt_{}'.format(f) ] = globals()[ '{}'.format(f) ][0].data # 데이터 정보

print(hdr_u)
print('--------------------------------------------------')
print(dt_u) 

In [ ]:
# 각 필터 fits 파일 데이터 통계 정보

for f in filter:
    print('데이터 구조_'+f+':', globals()[ 'dt_{}'.format(f) ].shape) # 행렬
    print('데이터 최소값_'+f+':', np.min( globals()[ 'dt_{}'.format(f)])) # 최소값
    print('데이터 최대값_'+f+':', np.max( globals()[ 'dt_{}'.format(f)])) # 최대값
    print('데이터 평균_'+f+':', np.mean( globals()[ 'dt_{}'.format(f)])) # 평균값
    print('데이터 중앙값_'+f+':', np.median( globals()[ 'dt_{}'.format(f)])) # 중앙값
    print('데이터 표준편차값_'+f+':', np.std( globals()[ 'dt_{}'.format(f) ])) # 표준준편차값
    print('--------------------------------------------')

### **이미지 시각화**

* 분석하려는 은하의 fits 파일에 담긴 적경, 적위 위치 좌표를 컴퓨터가 이미지로써 인식이 가능하도록 픽셀 좌표로 변환합니다. 
* fits 파일의 원본 전체 이미지 속에서 분석하려는 은하의 위치를 확인합니다. 

In [ ]:
# 각 필터 fits 파일 천체 이미지의 중심 픽셀 및 천문 좌표 정보

for f in filter:
    globals()[ 'wcs_{}'.format(f) ] = WCS(globals()[ 'hdr_{}'.format(f) ])
    print( 'wcs_'+f+':', globals()[ 'wcs_{}'.format(f) ] )
    print( '-----------------------------------------' )

In [ ]:
# 각 필터 fits 파일 관찰 대상의 천문 좌표를 픽셀 좌표로 변경한 정보

if ring_num == 1401: # 1401 고리은하의 천문 좌표 -> 픽셀 좌표 변환
    for f in filter:
        ob_wcs = [213.3, 8.8]
        globals()['ob_pix_{}'.format(f)] = [350,1037]
        print('ob_pix_'+f+':', globals()['ob_pix_{}'.format(f) ][0], globals()[ 'ob_pix_{}'.format(f) ][1])
elif ring_num == 1450: # 1450 고리은하의 천문 좌표 -> 픽셀 좌표 변환
    for f in filter:
        ob_wcs = [228.3, 5.7]
        globals()['ob_pix_{}'.format(f)] = [729, 1124]
        print('ob_pix_'+f+':', globals()['ob_pix_{}'.format(f) ][0], globals()[ 'ob_pix_{}'.format(f) ][1])
else: # 그 외 모든 고리은하 천문 좌표 -> 픽셀 좌표 변환   
    ob_wcs1 = ring_ra 
    ob_wcs2 = ring_dec
    print('적경(ra):', ob_wcs1)
    print('적위(dec):', ob_wcs2)
    ob_wcs = [ob_wcs1, ob_wcs2]
    for f in filter:
        globals()['ob_pix_{}'.format(f)] = np.around(globals()['wcs_{}'.format(f)].world_to_pixel_values(ob_wcs[0], ob_wcs[1] )).astype('int32')
        print('픽셀 좌표_'+f+':', globals()[ 'ob_pix_{}'.format(f)][0], globals()['ob_pix_{}'.format(f)][1])

In [ ]:
# 전체 이미지 시각화

for f in filter: # 이미지 밝기 백분율
    globals()['max_value_{}'.format(f)] = np.percentile(globals()[ 'dt_{}'.format(f) ], 99.8) # 최대갓 백분율
    globals()['min_value_{}'.format(f)] = np.percentile(globals()[ 'dt_{}'.format(f) ], 15) # 최솟값 백분율

fig = plt.figure(figsize = (20,18)) # 전체 이미지 크기
n = 1
for f in filter:
    ax = fig.add_subplot(4,3,n, projection = wcs_u) # 천문좌표 중심 이미지 시각화
    im1 = plt.imshow(globals()['dt_{}'.format(f)], cmap = 'gray_r', vmax = globals()['max_value_{}'.format(f)], vmin = globals()['min_value_{}'.format(f)], origin = 'lower') # 전체 이미지
    ax.scatter(ob_wcs[0], ob_wcs[1], transform = ax.get_transform('fk5'), s = 300, edgecolor = 'red', facecolor = 'none', linewidth = 2) # 고리은하 위치 표시
    plt.grid(color = 'white', ls = '--') 
    plt.colorbar(im1) 
    plt.title(f +'_ra '+str(ob_wcs[0])+', dec '+str(ob_wcs[1])) 
    
    ax = fig.add_subplot(4,3,n+6) # 픽셀 좌표 중심 이미지 시각화
    im2 = plt.imshow( globals()['dt_{}'.format(f)], cmap = 'gray_r', vmax = globals()['max_value_{}'.format(f)], vmin = globals()[ 'min_value_{}'.format(f)], origin = 'lower') # 전체 이미지
    plt.plot( globals()['ob_pix_{}'.format(f)][0], globals()[ 'ob_pix_{}'.format(f)][1], 'o', ms = 20, mec='red', mfc='none', linewidth=5) # 고리은하 위치 표시
    plt.grid(color = 'white', ls = '--') 
    plt.colorbar(im2) # 컬러바
    plt.title(f +'_pix '+str( globals()['ob_pix_{}'.format(f)][0])+', '+str( globals()['ob_pix_{}'.format(f)][1])) 
    n+=1
plt.show()

# **데이터 처리**

## **데이터 변환**

### **이미지 변환**

* 원본 전체 이미지 중 분석하려는 은하만 포착하고 시각화 합니다.
* 이때 분석하기에 적합하도록 은하 이미지를 300 x 300 픽셀 이하의 정사각형 구조로 적절히 입력합니다. 기본값을 통한 자동 조절을 원하시면 엔터를 누르십시오.

In [ ]:
# 원본 전체 이미지 중 분석 대상 은하만 포착 및 시각화

vmax = 90. 
vmin = 15. 
print('max_percetile:', vmax)
print('min_percetile:', vmin)

fig = plt.figure(figsize = (14,9))
n = 1
for f in filter:
    globals()['max_value_{}_temp'.format(f)] = np.percentile(globals()['dt_{}'.format(f) ], vmax)
    globals()['min_value_{}_temp'.format(f)] = np.percentile(globals()['dt_{}'.format(f) ], vmin)
    ax = fig.add_subplot( 2,3,n )
    plt.imshow(globals()['dt_{}'.format(f)], cmap = 'gray_r', vmax = globals()['max_value_{}'.format(f)], vmin = globals()['min_value_{}'.format(f)], origin = 'lower')
    plt.grid(color = 'white', ls = '--')
    plt.colorbar()
    plt.xlim(globals()['ob_pix_{}'.format(f) ][0]-150, globals()['ob_pix_{}'.format(f) ][0]+150) # 고리은하 중심 x축 300픽셀
    plt.ylim(globals()['ob_pix_{}'.format(f) ][1]-150, globals()['ob_pix_{}'.format(f) ][1]+150) # 고리은하 중심 y축 300픽셀
    plt.title(f+'_pix '+str(globals()['ob_pix_{}'.format(f)][0])+', '+str(globals()['ob_pix_{}'.format(f)][1]))
    n+=1
plt.show()

In [ ]:
#@title ② 분석 이미지 크기
shape_x = 300 #@param {type:"integer", min:100, max:500, step:10}
shape_y = shape_x
print("분석 이미지 픽셀 구조:", shape_x, shape_y)

print('------- 분석하려는 은하가 모퉁이에 있는 경우 컴퓨터의 자동계산에 의해 입력한 픽셀 구조 보다 작게 될 수 있습니다. --------')

all_shape = []    
for f in filter:
    for i in reversed(range(50, shape_x+10, 10)): # 은하가 모퉁이에 있으면 이미지가 짤리기 때문에, 이미지가 정사각형이 되게끔  x축과 y축이 같아 질때 까지 짝수 단위로 축소 
        globals()['ob_dt_{}'.format(f)] = globals()['dt_{}'.format(f)][globals()['ob_pix_{}'.format(f)][1] - int(i/2) : globals()['ob_pix_{}'.format(f) ][1] + int(i/2), globals()['ob_pix_{}'.format(f)][0] - int(i/2) : globals()['ob_pix_{}'.format(f)][0] + int(i/2)]
        if globals()['ob_dt_{}'.format(f)].shape[0] == globals()['ob_dt_{}'.format(f)].shape[1]:
            all_shape.append(globals()['ob_dt_{}'.format(f)].shape)
            break
            
shape1 = np.min(all_shape) 
shape2 = np.min(all_shape)

center = [int(shape1/2), int(shape2/2)] # 이미지의 중심 좌표 출력(이미지가 정사각형이니 x축 중심, y축 중심 같음)
print('초기 이미지 중심 좌표:', center)
    
for f in filter: # 탐색된 정사각형 행렬로 이미지 조정
    globals()['ob_dt_{}'.format(f)] = globals()['dt_{}'.format(f)][globals()['ob_pix_{}'.format(f)][1] - int(shape1/2) : globals()['ob_pix_{}'.format(f)][1] + int(shape1/2), globals()['ob_pix_{}'.format(f)][0] - int(shape2/2) : globals()['ob_pix_{}'.format(f)][0] + int(shape2/2)] 
    print('이미지 픽셀 구조_'+f+':', globals()[ 'ob_dt_{}'.format(f) ].shape) 
        
# 그림
vmax = 90.
vmin = 15.
print('max_percetile:', vmax)
print('min_percetile:', vmin)

fig = plt.figure(figsize = (14,9))
n = 1
for f in filter:
    globals()['max_value_{}'.format(f)] = np.percentile(globals()['ob_dt_{}'.format(f)], vmax)
    globals()['min_value_{}'.format(f)] = np.percentile(globals()['ob_dt_{}'.format(f)], vmin)
    ax = fig.add_subplot( 2,3,n )
    plt.imshow(globals()['ob_dt_{}'.format(f)], cmap = 'gray_r', vmax = globals()['max_value_{}'.format(f)], vmin = globals()['min_value_{}'.format(f)], origin = 'lower')
    plt.plot(center[1], center[0], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.colorbar()
    plt.title(f+'_pix '+str(center[0])+', '+str(center[1]))
    n+=1
plt.show()

## **배경 하늘의 밝기 제거(skysub)**

### **배경 하늘 평균 값 추정**

* 배경 하늘 값 추정은 두 가지 방식으로 이루어집니다.
* 첫 번째는 이미지를 36개의 구역으로 균등하게 나눠, 평균 값이 가장 적은 5개 구역을 선별하고 그것을 평균화하여 배경 하늘 값을 추정합니다. 
  이를 '5개 최소구역평균 배경하늘 값' 이라고 명명합니다.
* 두 번째는 이미지에서 신호 대 잡음비(Signal Noise Ratio, SNR)가 1.017배가 넘는 천체는 마스킹을 한 뒤 그 나머지를 평균화 하여 배경 하늘 값을 추정합니다.
  이를 '천체마스킹평균 배경하늘 값'이라고 명명합니다.  

In [ ]:
# 각 필터의 이미지를 36개의 구역으로 균등하게 나눠, 평균 값이 가장 적은 5개 구역을 선별, 5개 구역의 평균으로 배경 하늘 값 추정

f_1 = 'i'
for f in filter:
    globals()['partition_{}'.format(f)] = []
    globals()['noise_mean_{}'.format(f)] = []

n = 1
for f in filter:
    for j in range(0,6):
        for k in range(0,6):
            globals()['partition_noise_{}'.format(f)] = globals()['ob_dt_{}'.format(f)][int(shape1/6)*j : int(shape1/6)*(j+1), int(shape2/6)*k : int(shape2/6)*(k+1)]
            globals()['partition_noisemean_{}'.format(f)] = np.mean(globals()['partition_noise_{}'.format(f)])
            globals()['partition_{}'.format(f)].append(globals()['partition_noise_{}'.format(f)])
            globals()['noise_mean_{}'.format(f)].append(globals()['partition_noisemean_{}'.format(f)])
            n += 1

for f in filter:
    globals()['nosie_mean_least5_{}'.format(f)] = np.sort(globals()['noise_mean_{}'.format(f)][0:5])
    globals()['sky_nosie_mean_least5_mean_{}'.format(f)] = np.mean(globals()['nosie_mean_least5_{}'.format(f)])
    print('5개 구역의 평균 하늘 값_' + f + ':' , globals()['nosie_mean_least5_{}'.format(f)])
    print('5개 구역의 평균 하늘 값을 평균화 한 값_' + f + ':', globals()['sky_nosie_mean_least5_mean_{}'.format(f)])
    print('------------------------------------------')

In [ ]:
# I 필터 기준 이미지에서 snr이 1.017배가 넘는 천체는 마스킹을 한 뒤 그 나머지를 모두 배경 하늘 값으로 추정

# 배경 하늘 값 추출시 제외 영역
snr = 1.017
globals()['exc_ob_{}'.format(f_1)] = np.where( globals() [ 'ob_dt_{}'.format(f_1) ] >= globals()['sky_nosie_mean_least5_mean_{}'.format(f_1)] * snr )
print(str(f_1)+' filter 잡음 영역 평균'+'('+str(globals()['sky_nosie_mean_least5_mean_{}'.format(f_1)])+')'+'*SNR'+'('+str(snr)+'):', globals()['sky_nosie_mean_least5_mean_{}'.format(f_1)] * snr )
print( '-----------------------------------------' )

exc_ob_cor= globals()['exc_ob_{}'.format(f_1)]
exc_ob_rad= 10
print('제외한 천체(객체)들의 반경(픽셀):', exc_ob_rad)
print( '-----------------------------------------' ) 
        
# 배경 하늘 값 추출시 제외 영역 마스크 지정
exc_mask_reg = []
exc_aperture_reg = []
n = 1
for j,k in zip(exc_ob_cor[1], exc_ob_cor[0]):
    exc_center = PixCoord(j, k)
    #print('%d_[x,y,r]:'%n, j, k, exc_ob_rad )
    exc_aperture = CirclePixelRegion(exc_center, exc_ob_rad)
    exc_mask = exc_aperture.to_mask(mode = 'center')
    exc_aperture_reg.append(exc_aperture)
    exc_mask_reg.append(exc_mask.bbox)
    n += 1

# 데이터 사본(copy) 생성
for f in filter:
    globals()['ob_dt_{}_c'.format(f)] = globals()['ob_dt_{}'.format(f)].copy()

# 배경 하늘 값 추출시 제외한 영역 0 변환(마스킹) 및 배경 하늘 값 평균
for f in filter:
    globals()['ob_dt_{}_c'.format(f)] = globals()['ob_dt_{}'.format(f)].copy()
    for i in range(len(exc_mask_reg)): 
        globals()['ob_dt_{}_c'.format(f)][exc_mask_reg[i].iymin : exc_mask_reg[i].iymax + 1, exc_mask_reg[i].ixmin : exc_mask_reg[i].ixmax + 1] = 0.
        globals()['sky_{}'.format(f)] = np.mean(np.true_divide(globals()['ob_dt_{}_c'.format(f)].sum(1),(globals()['ob_dt_{}_c'.format(f)]!= 0.).sum(1)))
    print('하늘 값_' + f + ':', globals()['sky_{}'.format(f)])
print('--------------------------------------')    
    
vmax = 90.
vmin = 15.
print('max_percetile:', vmax)
print('min_percetile:', vmin)
    
# 그림(i필터만)
fig = plt.figure( figsize = (10,4) )
globals()[ 'max_value_{}_temp'.format(f_1) ] = np.percentile( globals()[ 'ob_dt_{}'.format(f_1) ], vmax )
globals()[ 'min_value_{}_temp'.format(f_1) ] = np.percentile( globals()[ 'ob_dt_{}'.format(f_1) ], vmin )
ax = fig.add_subplot(1,2,1)
plt.imshow(globals()['ob_dt_{}'.format(f_1)], cmap = 'gray_r', vmax = globals()['max_value_{}_temp'.format(f_1)], vmin = globals()[ 'min_value_{}_temp'.format(f_1)], origin = 'lower')
plt.plot(globals()['exc_ob_{}'.format(f_1)][1], globals()['exc_ob_{}'.format(f_1)][0], 'o', ms = 2, mec='red', mfc='red', linewidth=1)
plt.grid(color = 'white', ls = '--')
plt.colorbar()
plt.title(f_1 + '_exception_region')

ax = fig.add_subplot(1,2,2)
plt.imshow(globals()['ob_dt_{}'.format(f_1)], cmap = 'gray_r', vmax = globals()['max_value_{}_temp'.format(f_1)], vmin = globals()[ 'min_value_{}_temp'.format(f_1)], origin = 'lower')
for m, o in zip(range(len(exc_mask_reg)), range(len(exc_aperture_reg))):
    ax.add_artist(exc_mask_reg[m].as_artist(facecolor='none', edgecolor='white', linewidth=2))
    ax.add_artist(exc_aperture_reg[o].as_artist(facecolor='none', edgecolor='orange', linewidth=2))  
plt.grid(color = 'white', ls = '--')
plt.colorbar()
plt.title(f_1 + '_exception_region_mask')
plt.show()

### **배경 하늘의 밝기가 제거된 하늘 값 추출**

* 배경 하늘의 밝기가 제거된 하늘 값 추출(skysub)은 은하만의 고유 밝기를 알기 위한 작업입니다. 이를 위해 '원본 이미지'에서 '배경 하늘 값'을 빼며 알고리즘은 다음의 규칙으로 작동합니다.
* 먼저 '원본 이미지'와 '천체마스킹평균 배경하늘 값'을 뺀 것을 우선으로 사용합니다.
* 그런데 그 과정에서 Nan이 발생하면(너무 많이 마스킹하면 Nan 발생), '원본 이미지'와 '5개 최소구역평균 배경하늘 값'을 뺀 것을 사용합니다.

In [ ]:
# 배경 하늘의 밝기가 제거된 하늘 값 추출(skysub)

my_skysub_u = []
my_skysub_g = []
my_skysub_r = []
my_skysub_i = []
my_skysub_z = []

if math.isnan(globals()['sky_{}'.format(f)]) == False:
    for f in filter: # 배경 하늘의 밝기가 제거된 하늘 값 추출시 제외영역의 통계값이 nan 값이 아니면, '천체마스킹평균 배경하늘 값'을 배경 하늘 평균값으로 사용 
        globals()['skysub_{}'.format(f)] = (globals()['ob_dt_{}'.format(f)] - globals()['sky_{}'.format(f)]) # 배경 하늘의 밝기가 제거된 하늘 값 = 원본 값 - 배경 하늘 평균값
        print("배경 하늘의 밝기가 제거된 하늘 값 추출시 제외 영역의 통계값이 nan 값이 아니므로 " + str(f) + " 필터의 천체마스킹평균 배경하늘 값이 이 계산에 사용됨")
else:
    for f in filter: # 배경 하늘의 밝기가 제거된 하늘 값 추출시 제외영역의 통계값이 nan 값이면, '5개 최소구역평균 배경하늘 값' 을  배경 하늘 평균값으로 사용
        globals()['skysub_{}'.format(f)] = (globals()['ob_dt_{}'.format(f)] - globals()['sky_nosie_mean_least5_mean_{}'.format(f)]) # 배경 하늘의 밝기가 제거된 하늘 값 = 원본 값 - 배경 하늘 평균값
        print("배경 하늘의 밝기가 제거된 하늘 값 추출시 제외 영역의 통계값이 nan 값이므로 "+ str(f) + " 필터의 5개 최소구역평균 배경하늘 값이 이 계산에 사용됨")

for f in filter:
    if f == 'u':
        my_skysub_u.append(np.min(globals()['skysub_{}'.format(f)]))
        my_skysub_u.append(np.max(globals()['skysub_{}'.format(f)]))
        my_skysub_u.append(np.mean(globals()['skysub_{}'.format(f)]))
        my_skysub_u.append(np.median(globals()['skysub_{}'.format(f)]))
        my_skysub_u.append(np.std(globals()['skysub_{}'.format(f)]))
    elif f == 'g':
        my_skysub_g.append(np.min(globals()['skysub_{}'.format(f)]))
        my_skysub_g.append(np.max(globals()['skysub_{}'.format(f)]))
        my_skysub_g.append(np.mean(globals()['skysub_{}'.format(f)]))
        my_skysub_g.append(np.median(globals()['skysub_{}'.format(f)]))
        my_skysub_g.append(np.std(globals()['skysub_{}'.format(f)]))
    elif f == 'r':
        my_skysub_r.append(np.min(globals()['skysub_{}'.format(f)]))
        my_skysub_r.append(np.max(globals()['skysub_{}'.format(f)]))
        my_skysub_r.append(np.mean(globals()['skysub_{}'.format(f)]))
        my_skysub_r.append(np.median(globals()['skysub_{}'.format(f)]))
        my_skysub_r.append(np.std(globals()['skysub_{}'.format(f)]))      
    elif f == 'i':
        my_skysub_i.append(np.min(globals()['skysub_{}'.format(f)]))
        my_skysub_i.append(np.max(globals()['skysub_{}'.format(f)]))
        my_skysub_i.append(np.mean(globals()['skysub_{}'.format(f)]))
        my_skysub_i.append(np.median(globals()['skysub_{}'.format(f)]))
        my_skysub_i.append(np.std(globals()['skysub_{}'.format(f)]))
    else:
        my_skysub_z.append(np.min(globals()['skysub_{}'.format(f)]))
        my_skysub_z.append(np.max(globals()['skysub_{}'.format(f)]))
        my_skysub_z.append(np.mean(globals()['skysub_{}'.format(f)]))
        my_skysub_z.append(np.median(globals()['skysub_{}'.format(f)]))
        my_skysub_z.append(np.std(globals()['skysub_{}'.format(f)]))

In [ ]:
# 은하만의 밝기 산출 및 데이터 프레임 변환 및 csv 저장

# 은하만의 밝기 산출 및 데이터 프레임 변환
all_skysub = {
    'min' : [my_skysub_u[0], my_skysub_g[0], my_skysub_r[0], my_skysub_i[0], my_skysub_z[0]], # 각 필터별 하늘의 상대적 세기 최소값
    'max' : [my_skysub_u[1], my_skysub_g[1], my_skysub_r[1], my_skysub_i[1], my_skysub_z[1]], # 각 필터별 하늘의 상대적 세기 최대값
    'mean' : [my_skysub_u[2], my_skysub_g[2], my_skysub_r[2], my_skysub_i[2], my_skysub_z[2]], # 각 필터별 하늘의 상대적 세기 평균값
    'median' : [my_skysub_u[3], my_skysub_g[3], my_skysub_r[3], my_skysub_i[3], my_skysub_z[3]], # 각 필터별 하늘의 상대적 세기 중앙값
    'stdev' : [my_skysub_u[4], my_skysub_g[4], my_skysub_r[4], my_skysub_i[4], my_skysub_z[4]], # 각 필터별 하늘의 상대적 세기 표준편차값
    }
columns = ['min', 'max', 'mean', 'median', 'stdev']
index = ['u', 'g', 'r', 'i', 'z']
skysub_frame = pd.DataFrame(all_skysub, index = index, columns=columns).round(3)
skysub_frame.index.name ='filter'
display(skysub_frame)

# csv 파일 저장
save_file_name = str(ring_num)+'_1_skysub.csv'
skysub_frame.to_csv(save_path + save_file_name, header=True, index=False)

In [ ]:
# 배경 하늘의 밝기가 제거된 하늘 값 추출 그림 시각화 

vmax = 90.
vmin = 15.
print('max_percetile:', vmax)
print('min_percetile:', vmin)

# 그림
fig = plt.figure( figsize = (14,9) )  
n = 1
for f in filter:
    globals()['max_value_{}'.format(f)] = np.percentile(globals()['skysub_{}'.format(f)], vmax)
    globals()['min_value_{}'.format(f)] = np.percentile(globals()['skysub_{}'.format(f)], vmin)
    ax = fig.add_subplot(2,3,n)
    plt.imshow(globals()['skysub_{}'.format(f)], cmap = 'gray_r', vmax = globals()['max_value_{}'.format(f)], vmin = globals()['min_value_{}'.format(f)], origin = 'lower')
    plt.plot(center[1], center[0], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.colorbar()
    plt.title(f + '_sky_sub')
    n += 1
plt.show()

## **데이터 다림질(Smoothing)**

### **커털 표준편차와 모드를 사용한 다림질 옵션 조정**

* 데이터 다림질은 잡음과 섞인 이미지의 신호를 정규분포 형태로 부드럽게 바꿔주는 작업입니다.
* 다림질의 옵션에는 가우시안 2D커널의 가우시안 표준편차와 커널 모드가 있습니다. 기본값으로 진행할시 엔터를 누르십시오.
* 가우시안 표준편차가 클수록 등광도선이 더 부드럽고 단순해집니다.
* 가우시안 커널 모드 포함 다림질에 대한 자세한 설명은 다음의 주소에서 확인하십시오. https://docs.astropy.org/en/stable/api/astropy.convolution.Gaussian2DKernel.html

In [ ]:
# 데이터 다림질 및 그 결과 데이터 프레임 변환 csv로 저장하는 함수 정의
def smoothing(kernel_stddev = 0.9, kernel_mode = 'center', x_size = None, y_size = None, sub_u = skysub_u, sub_g = skysub_g, sub_r = skysub_r, sub_i = skysub_i, sub_z = skysub_z):

    # 다림질
    kernel = Gaussian2DKernel(kernel_stddev, mode = kernel_mode, x_size=x_size, y_size=y_size) #  다림질을 위한 가우시안2D 커널 표준편차 및 모드 지정   
    print('가우시안2D 커널 표준편차:', kernel_stddev) #  다림질을 위한 가우시안2D 커널 표준편차
    print('가우시안2D 커널 모드:', kernel_mode) # 다림질을 위한 커널 모드, 커널 종류: 'linear_interp', 'oversample', 'integrate'

    # 결과
    my_smo_u = []
    my_smo_g = []
    my_smo_r = []
    my_smo_i = []
    my_smo_z = []
 
    smo_u = convolve(sub_u, kernel, nan_treatment='interpolate')
    my_smo_u.append(np.min(smo_u))
    my_smo_u.append(np.max(smo_u))
    my_smo_u.append(np.mean(smo_u))
    my_smo_u.append(np.median(smo_u))
    my_smo_u.append(np.std(smo_u))
    
    smo_g = convolve(sub_g, kernel, nan_treatment='interpolate')
    my_smo_g.append(np.min(smo_g))
    my_smo_g.append(np.max(smo_g))
    my_smo_g.append(np.mean(smo_g))
    my_smo_g.append(np.median(smo_g))
    my_smo_g.append(np.std(smo_g))    

    smo_r = convolve(sub_r, kernel, nan_treatment='interpolate')
    my_smo_r.append(np.min(smo_r))
    my_smo_r.append(np.max(smo_r))
    my_smo_r.append(np.mean(smo_r))
    my_smo_r.append(np.median(smo_r))
    my_smo_r.append(np.std(smo_r))    
     
        
    smo_i = convolve(sub_i, kernel, nan_treatment='interpolate')
    my_smo_i.append(np.min(smo_i))
    my_smo_i.append(np.max(smo_i))
    my_smo_i.append(np.mean(smo_i))
    my_smo_i.append(np.median(smo_i))
    my_smo_i.append(np.std(smo_i))
 
    smo_z = convolve(sub_z, kernel, nan_treatment='interpolate')
    my_smo_z.append(np.min(smo_z))
    my_smo_z.append(np.max(smo_z))
    my_smo_z.append(np.mean(smo_z))
    my_smo_z.append(np.median(smo_z))
    my_smo_z.append(np.std(smo_z))  
        
    # 데이터 다림질 값 산출 및 데이터 프레임 변환
    all_smo = {
        'min' : [my_smo_u[0], my_smo_g[0], my_smo_r[0], my_smo_i[0], my_smo_z[0]], # 각 필터별 다림질 최소값
        'max' : [my_smo_u[1], my_smo_g[1], my_smo_r[1], my_smo_i[1], my_smo_z[1]], # 각 필터별 다림질 최대값
        'mean' : [my_smo_u[2], my_smo_g[2], my_smo_r[2], my_smo_i[2], my_smo_z[2]], # 각 필터별 다림질 평균값
        'median' : [my_smo_u[3], my_smo_g[3], my_smo_r[3], my_smo_i[3], my_smo_z[3]], # 각 필터별 다림질 중앙값
        'stdev' : [my_smo_u[4], my_smo_g[4], my_smo_r[4], my_smo_i[4], my_smo_z[4]], # 각 필터별 다림질 표준편차값
        }
    columns = ['min', 'max', 'mean', 'median', 'stdev']
    index = ['u', 'g', 'r', 'i', 'z']
    smo_frame = pd.DataFrame(all_smo, index = index, columns=columns).round(3)
    smo_frame.index.name ='filter'
    display(smo_frame)
    
    # csv 파일 저장
    save_file_name = str(ring_num)+'_2_smo.csv'
    smo_frame.to_csv(save_path + save_file_name, header=True, index=False)
    
    return smo_u, smo_g, smo_r, smo_i, smo_z

In [ ]:
#@title ③ Smoothing 옵션
kn_st = 0.9 #@param {type:"number"}
kn_md = "center" #@param ["center", "linear_interp", "oversample", "integrate"]

smo_u, smo_g, smo_r, smo_i, smo_z = smoothing(kernel_stddev=kn_st, kernel_mode=kn_md)
print("✅ Smoothing 완료")


In [ ]:
# 다림질 그림 시각화

vmax = 90.
vmin = 15.
print('max_percetile:', vmax)
print('min_percetile:', vmin)

# 그림
fig = plt.figure( figsize = (14,9) ) 
n = 1
for f in filter:
    globals()['max_value_{}'.format(f)] = np.percentile(globals()['smo_{}'.format(f)], vmax)
    globals()['min_value_{}'.format(f)] = np.percentile(globals()['smo_{}'.format(f)], vmin)

    ax = fig.add_subplot(2,3,n)
    plt.imshow(globals()['smo_{}'.format(f)], cmap = 'gray_r', vmax = globals()['max_value_{}'.format(f)], vmin = globals()['min_value_{}'.format(f)], origin = 'lower')
    plt.plot(center[1], center[0], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.colorbar()
    plt.title(f + '_smoothing')
    n += 1
plt.show()

# **데이터 분석**

## **타원 기하학 산출**

### **이미지 중심좌표 보정 및 등광도선 설정**

* 각 필터에서 `photutils.EllipseGeometry.find_center()`를 사용해 중심을 자동 추정합니다.
* 원본 Notebook처럼 중심좌표를 다시 손으로 입력하지 않아도 됩니다.


In [ ]:
# 각 필터별 이미지 중심 좌표 자동 추정
cen_u, cen_g, cen_r, cen_i, cen_z = [], [], [], [], []

for f in filter:
    geom = EllipseGeometry(x0=center[1], y0=center[0], sma=20, eps=0.5, pa=-0.5)
    try:
        geom.find_center(globals()['smo_{}'.format(f)])
        cx, cy = float(geom.x0), float(geom.y0)
    except Exception:
        cx, cy = float(center[1]), float(center[0])
    globals()['cen_{}'.format(f)].extend([cx, cy])
    print(f"{f}-band 중심좌표 (x, y): ({cx:.2f}, {cy:.2f})")


* 자동 추정된 중심좌표를 그대로 사용합니다. 필요하면 아래 셀의 값을 직접 수정할 수 있습니다.


In [ ]:
# 중심좌표 확인
for f in filter:
    print(f, globals()['cen_{}'.format(f)])


* 등광도선의 최소 백분위와 레벨을 조정합니다. 기본값으로 진행할시 엔터를 누르십시오.
* 등광도선의 최소 백분위가 낮을수록 등광도선이 더 자세하게, 레벨은 높을수록 더 많이 등광도선이 그려집니다.
* 은하 내부에 천체가 있는지, 은하 내부가 복잡한지 등 등광도선의 모습을 통해 은하의 상태를 확인합니다.
* 은하 내부에 천체가 있는 이유는, 사실 시선 방향에 있던 별과 겹쳐서 마치 은하 내부에 천체가 있는 것처럼 이미지에 투영되었기 때문입니다. 즉, 은하 내부에는 별이 없습니다.
* 은하 내부가 복잡한 이유는, 은하를 희미한 밝기를 더 많이 포착하기 위해 등광도선을 많이 그렸기 때문입니다. 

In [ ]:
#@title ④ 등광도선 표시 옵션
min_per = 89 #@param {type:"integer", min:50, max:99}
step_num = 25 #@param {type:"integer", min:5, max:60}
max_per = 99.9

print('최소 백분위:', min_per)
print('최대 백분위:', max_per)
print('등광도선 레벨:', step_num)

for f in filter:
    globals()['levels_{}'.format(f)] = np.linspace(
        np.percentile(globals()['smo_{}'.format(f)], min_per),
        np.percentile(globals()['smo_{}'.format(f)], max_per),
        step_num
    )


In [ ]:
# 이미지 등광도선 확인
def image_contour_identification():
    vmax = 90.
    vmin = 15.

    for f in filter:
        globals()['max_value_{}_temp'.format(f)] = np.percentile( globals()['smo_{}'.format(f)], vmax)
        globals()['min_value_{}_temp'.format(f)] = np.percentile( globals()['smo_{}'.format(f)], vmin)
        globals()['max_{}'.format(f)] = np.max(globals()['smo_{}'.format(f)])

    # 그림
    fig = plt.figure( figsize = (17, 8) )

    ax1 = fig.add_subplot(2,4,1)
    im = plt.imshow(globals()['smo_{}'.format(filter[1])], cmap = 'gray_r', vmax = globals()['max_value_{}_temp'.format(filter[1])], vmin = globals()[ 'min_value_{}_temp'.format(filter[1])], origin = 'lower') #interpolation='nearest'
    plt.colorbar(im)
    ct = plt.contour(globals()['smo_{}'.format(filter[1])], levels = globals()['levels_{}'.format(filter[1])], origin = 'lower', colors = 'aqua', linewidths = 1.0)
    plt.plot(globals()['cen_{}'.format(filter[1])][0], globals()['cen_{}'.format(filter[1])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[1] + '_reg')

    ax2 = fig.add_subplot(2,4,2)
    ct = plt.contour(globals()['smo_{}'.format(filter[1])], levels = globals()['levels_{}'.format(filter[1])], origin = 'lower', linewidths = 0.5, colors = 'black') #colors= colors,
    #plt.xlim(globals()['cen_{}'.format(filter[1])][0]-50, globals()['cen_{}'.format(filter[1])][0]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    #plt.ylim(globals()['cen_{}'.format(filter[1])][1]-50, globals()['cen_{}'.format(filter[1])][1]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    plt.plot(globals()['cen_{}'.format(filter[1])][0], globals()['cen_{}'.format(filter[1])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[1] + '_reg, min_per: ' + str(min_per) + ', max_per: ' + str(max_per) + ', step: ' + str(step_num))
    plt.show()

    ax3 = fig.add_subplot(2,4,3)
    im = plt.imshow(globals()['smo_{}'.format(filter[2])], cmap = 'gray_r', vmax = globals()['max_value_{}_temp'.format(filter[2])], vmin = globals()[ 'min_value_{}_temp'.format(filter[2])], origin = 'lower') #interpolation='nearest'
    plt.colorbar(im)
    ct = plt.contour(globals()['smo_{}'.format(filter[2])], levels = globals()['levels_{}'.format(filter[2])], origin = 'lower', colors = 'aqua', linewidths = 1.0)
    plt.plot(globals()['cen_{}'.format(filter[2])][0], globals()['cen_{}'.format(filter[2])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[2] + '_reg')

    ax4 = fig.add_subplot(2,4,4)
    ct = plt.contour(globals()['smo_{}'.format(filter[2])], levels = globals()['levels_{}'.format(filter[2])], origin = 'lower', linewidths = 0.5, colors = 'black') #colors= colors,
    #plt.xlim(globals()['cen_{}'.format(filter[2])][0]-50, globals()['cen_{}'.format(filter[2])][0]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    #plt.ylim(globals()['cen_{}'.format(filter[2])][1]-50, globals()['cen_{}'.format(filter[2])][1]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    plt.plot( globals()['cen_{}'.format(filter[2])][0],  globals()['cen_{}'.format(filter[2])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[2] + '_reg, min_per: ' + str(min_per) + ', max_per: ' + str(max_per) + ', step: ' + str(step_num))
    plt.show()

    ax5 = fig.add_subplot(2,4,5)
    im = plt.imshow(globals()['smo_{}'.format(filter[3])], cmap = 'gray_r', vmax = globals()['max_value_{}_temp'.format(filter[3])], vmin = globals()[ 'min_value_{}_temp'.format(filter[3])], origin = 'lower') #interpolation='nearest'
    plt.colorbar(im)
    ct = plt.contour(globals()['smo_{}'.format(filter[3])], levels = globals()['levels_{}'.format(filter[3])], origin = 'lower', colors = 'aqua', linewidths = 1.0)
    plt.plot( globals()['cen_{}'.format(filter[3])][0], globals()['cen_{}'.format(filter[3])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[3] + '_reg')

    ax6 = fig.add_subplot(2,4,6)
    ct = plt.contour(globals()['smo_{}'.format(filter[3])], levels = globals()['levels_{}'.format(filter[3])], origin = 'lower', linewidths = 0.5, colors = 'black') #colors= colors,
    #plt.xlim(globals()['cen_{}'.format(filter[3])][0]-50, globals()['cen_{}'.format(filter[3])][0]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    #plt.ylim(globals()['cen_{}'.format(filter[3])][1]-50, globals()['cen_{}'.format(filter[3])][1]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    plt.plot(globals()['cen_{}'.format(filter[3])][0], globals()['cen_{}'.format(filter[3])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[3] + '_reg, min_per: ' + str(min_per) + ', max_per: ' + str(max_per) + ', step: ' + str(step_num))
    plt.show()

    ax7 = fig.add_subplot(2,4,7)
    im = plt.imshow(globals()['smo_{}'.format(filter[4])], cmap = 'gray_r', vmax = globals()['max_value_{}_temp'.format(filter[4])], vmin = globals()[ 'min_value_{}_temp'.format(filter[4])], origin = 'lower') #interpolation='nearest'
    plt.colorbar(im)
    ct = plt.contour(globals()['smo_{}'.format(filter[4])], levels = globals()['levels_{}'.format(filter[4])], origin = 'lower', colors = 'aqua', linewidths = 1.0)
    plt.plot( globals()['cen_{}'.format(filter[4])][0], globals()['cen_{}'.format(filter[4])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[4] + '_reg')

    ax8 = fig.add_subplot(2,4,8)
    ct = plt.contour(globals()['smo_{}'.format(filter[4])], levels = globals()['levels_{}'.format(filter[4])], origin = 'lower', linewidths = 0.5, colors = 'black') #colors= colors,
    #plt.xlim(globals()['cen_{}'.format(filter[4])][0]-50, globals()['cen_{}'.format(filter[4])][0]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    #plt.ylim(globals()['cen_{}'.format(filter[4])][1]-50, globals()['cen_{}'.format(filter[4])][1]+50) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    plt.plot(globals()['cen_{}'.format(filter[4])][0], globals()['cen_{}'.format(filter[4])][1], '+', ms = 10, mec='red', mfc='red', linewidth=2)
    plt.grid(color = 'white', ls = '--')
    plt.title(filter[4] + '_reg, min_per: ' + str(min_per) + ', max_per: ' + str(max_per) + ', step: ' + str(step_num))
    plt.show()
    
image_contour_identification()

In [ ]:
# 이미지 저장

save_file_name = str(ring_num)+'_0_image_contour_identification.jpg'
plt.savefig(save_path + save_file_name)

### **내부 천체 마스킹 — 선택 사항**

Colab 간편판의 첫 실행에서는 이 단계를 건너뛰는 것을 권장합니다.  
정밀 분석이 필요한 경우 원본 Notebook의 수동 마스킹 과정을 이용할 수 있습니다.


* 기본 실행에서는 추가 마스킹을 하지 않고 다음 단계로 이동합니다.


In [ ]:
# Colab 간편판: 첫 실행에서는 추가 천체 마스킹 생략
select = 'n'
print("추가 천체 마스킹: 생략")
print("다음 단계에서 i-band 영상을 보고 타원의 초기값을 정합니다.")


### **초기 타원 기하학 설정 — 이미지에서 직접 클릭**


원본 연구의 **이미지 직접 클릭 방식**을 Colab에서도 사용합니다.

Colab에서는 `%matplotlib widget` 백엔드가 Python/Colab 버전에 따라 동작하지 않는 경우가 있어,  
이 간편판에서는 **브라우저 화면의 실제 마우스 클릭 좌표를 JavaScript로 받아 Python 좌표로 변환**합니다.

### 사용 방법
1. 아래 셀을 실행합니다.
2. 표시된 i-band 은하 영상에서 **첫 번째 왼쪽 클릭 = 장축 방향 끝점**을 선택합니다.
3. **두 번째 왼쪽 클릭 = 단축 방향 끝점**을 선택합니다.
4. 잘못 클릭했다면 **오른쪽 클릭 또는 ‘초기화’ 버튼**으로 지웁니다.
5. 두 점이 맞으면 **‘선택 확정’** 버튼을 누릅니다.
6. 장축 반경·단축 반경·타원율·위치각이 자동 계산되고, 녹색 타원이 결과에 표시됩니다.

> 숫자로 장축·단축·위치각을 직접 입력할 필요가 없습니다.


In [ ]:
#@title ⑤ i-band 영상에서 장축·단축 경계점 직접 클릭

def _array_to_click_png(arr, center_xy, vmin_percentile=15, vmax_percentile=90):
    """origin='lower' 기준으로 보이는 grayscale PNG를 data URL로 변환합니다."""
    arr = np.asarray(arr, dtype=float)
    lo = np.nanpercentile(arr, vmin_percentile)
    hi = np.nanpercentile(arr, vmax_percentile)
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0

    norm = np.clip((arr - lo) / (hi - lo), 0, 1)
    # 원본 Notebook의 gray_r 느낌: 밝은 값일수록 어둡게
    gray = ((1.0 - norm) * 255).astype(np.uint8)
    rgb = np.dstack([gray, gray, gray])

    # 브라우저 이미지는 위가 0이므로, matplotlib origin='lower'와 같게 보이도록 상하 반전
    rgb_show = np.flipud(rgb).copy()

    # 중심점 빨간 십자 표시
    cx, cy = center_xy
    h, w = arr.shape
    px = int(round(cx))
    py_screen = int(round((h - 1) - cy))
    if 0 <= px < w and 0 <= py_screen < h:
        for d in range(-7, 8):
            if 0 <= px+d < w:
                rgb_show[py_screen, px+d] = [255, 45, 45]
            if 0 <= py_screen+d < h:
                rgb_show[py_screen+d, px] = [255, 45, 45]

    im = Image.fromarray(rgb_show)
    buf = BytesIO()
    im.save(buf, format="PNG")
    data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")
    return data_url, w, h


def _colab_two_point_click(image_array, center_xy):
    data_url, data_w, data_h = _array_to_click_png(image_array, center_xy)
    cx, cy = float(center_xy[0]), float(center_xy[1])

    display(Javascript(r"""
    async function astronomyTwoPointClick(dataUrl, dataW, dataH, cx, cy) {
      return new Promise((resolve) => {
        const old = document.getElementById('astro-click-selector');
        if (old) old.remove();

        const box = document.createElement('div');
        box.id = 'astro-click-selector';
        box.style.cssText = 'font-family:Arial,sans-serif;padding:12px;border:1px solid #bbb;border-radius:10px;display:inline-block;background:white;color:#111;';

        const msg = document.createElement('div');
        msg.style.cssText = 'margin-bottom:8px;font-weight:600;';
        msg.textContent = '① 장축 방향 끝점 클릭 → ② 단축 방향 끝점 클릭';
        box.appendChild(msg);

        const canvas = document.createElement('canvas');
        canvas.width = dataW;
        canvas.height = dataH;
        canvas.style.cssText = 'display:block;max-width:min(720px,95vw);height:auto;border:1px solid #777;cursor:crosshair;';
        box.appendChild(canvas);

        const help = document.createElement('div');
        help.style.cssText = 'margin-top:8px;font-size:13px;color:#333;';
        help.textContent = '좌클릭: 점 선택 | 우클릭: 초기화';
        box.appendChild(help);

        const controls = document.createElement('div');
        controls.style.marginTop = '8px';

        const resetBtn = document.createElement('button');
        resetBtn.textContent = '초기화';
        resetBtn.style.marginRight = '8px';

        const okBtn = document.createElement('button');
        okBtn.textContent = '선택 확정';
        okBtn.disabled = true;

        controls.appendChild(resetBtn);
        controls.appendChild(okBtn);
        box.appendChild(controls);

        document.body.appendChild(box);

        const ctx = canvas.getContext('2d');
        const img = new Image();
        let pts = [];

        function dataToScreen(pt) {
          return [pt[0], (dataH - 1) - pt[1]];
        }

        function redraw() {
          ctx.clearRect(0, 0, dataW, dataH);
          ctx.drawImage(img, 0, 0, dataW, dataH);

          // 중심점
          const c = dataToScreen([cx, cy]);
          ctx.strokeStyle = '#ff2d2d';
          ctx.lineWidth = 2;
          ctx.beginPath();
          ctx.moveTo(c[0]-8, c[1]); ctx.lineTo(c[0]+8, c[1]);
          ctx.moveTo(c[0], c[1]-8); ctx.lineTo(c[0], c[1]+8);
          ctx.stroke();

          const colors = ['#00ff66', '#ff9f1a'];
          pts.forEach((p, idx) => {
            const s = dataToScreen(p);
            ctx.strokeStyle = colors[idx];
            ctx.fillStyle = colors[idx];
            ctx.lineWidth = 2;
            ctx.beginPath();
            ctx.moveTo(c[0], c[1]);
            ctx.lineTo(s[0], s[1]);
            ctx.stroke();
            ctx.beginPath();
            ctx.arc(s[0], s[1], 5, 0, Math.PI*2);
            ctx.fill();
          });

          if (pts.length === 0) {
            msg.textContent = '① 장축 방향 끝점을 클릭하세요.';
          } else if (pts.length === 1) {
            msg.textContent = '② 단축 방향 끝점을 클릭하세요.';
          } else {
            msg.textContent = '두 점 선택 완료. 맞으면 “선택 확정”, 다시 하려면 초기화하세요.';
          }
          okBtn.disabled = pts.length !== 2;
        }

        function eventToData(e) {
          const rect = canvas.getBoundingClientRect();
          const sx = (e.clientX - rect.left) * dataW / rect.width;
          const sy = (e.clientY - rect.top) * dataH / rect.height;
          const x = sx;
          const y = (dataH - 1) - sy;
          return [x, y];
        }

        canvas.addEventListener('click', (e) => {
          if (pts.length >= 2) return;
          pts.push(eventToData(e));
          redraw();
        });

        canvas.addEventListener('contextmenu', (e) => {
          e.preventDefault();
          pts = [];
          redraw();
        });

        resetBtn.onclick = () => {
          pts = [];
          redraw();
        };

        okBtn.onclick = () => {
          if (pts.length !== 2) return;
          const out = pts;
          box.remove();
          resolve(out);
        };

        img.onload = redraw;
        img.src = dataUrl;
      });
    }
    """))

    points = eval_js(
        "astronomyTwoPointClick(" +
        json.dumps(data_url) + "," +
        str(data_w) + "," +
        str(data_h) + "," +
        str(cx) + "," +
        str(cy) + ")"
    )
    return np.asarray(points, dtype=float)


f_1 = 'i'
cen = globals()['cen_{}'.format(f_1)]

clicked = _colab_two_point_click(globals()['smo_{}'.format(f_1)], cen)
major_point = clicked[0]
minor_point = clicked[1]

major_ax = float(np.hypot(major_point[0] - cen[0], major_point[1] - cen[1]))
minor_ax = float(np.hypot(minor_point[0] - cen[0], minor_point[1] - cen[1]))

# 혹시 두 번째 점이 더 멀게 찍혔으면 사용자 의도 보존을 위해 경고 후 자동 교환
if minor_ax > major_ax:
    print("⚠️ 두 번째 점이 첫 번째 점보다 중심에서 더 멉니다. 장축/단축을 자동 교환합니다.")
    major_point, minor_point = minor_point.copy(), major_point.copy()
    major_ax, minor_ax = minor_ax, major_ax

position_angle_rad = float(np.arctan2(
    major_point[1] - cen[1],
    major_point[0] - cen[0]
))
position_angle_deg = float(np.degrees(position_angle_rad))
ellipse = float((major_ax - minor_ax) / major_ax)

print("✅ 마우스 클릭 좌표 선택 완료")
print("장축 끝점:", np.round(major_point, 2))
print("단축 끝점:", np.round(minor_point, 2))
print(f"장축 반경: {major_ax:.2f} px")
print(f"단축 반경: {minor_ax:.2f} px")
print(f"타원율: {ellipse:.4f}")
print(f"위치각: {position_angle_deg:.2f}°")


In [ ]:
# 클릭 결과를 그림 위에서 확인
plt.figure(figsize=(7,7))
arr = globals()['smo_{}'.format(f_1)]
vmax_i = np.percentile(arr, 90)
vmin_i = np.percentile(arr, 15)

plt.imshow(arr, cmap='gray_r', vmax=vmax_i, vmin=vmin_i, origin='lower')
plt.contour(
    arr,
    levels=globals()['levels_{}'.format(f_1)],
    origin='lower',
    colors='aqua',
    linewidths=0.8
)
plt.plot(cen[0], cen[1], '+', ms=14, mec='red', mfc='red', mew=2)
plt.plot(major_point[0], major_point[1], 'o', color='lime', ms=8, label='major click')
plt.plot(minor_point[0], minor_point[1], 'o', color='orange', ms=8, label='minor click')
plt.plot([cen[0], major_point[0]], [cen[1], major_point[1]], color='lime', lw=2)
plt.plot([cen[0], minor_point[0]], [cen[1], minor_point[1]], color='orange', lw=2)

preview = patches.Ellipse(
    (cen[0], cen[1]),
    width=2*major_ax,
    height=2*minor_ax,
    angle=position_angle_deg,
    fill=False,
    edgecolor='greenyellow',
    linewidth=2.5,
    linestyle='--'
)
plt.gca().add_patch(preview)
plt.grid(color='white', ls='--', alpha=0.35)
plt.legend(loc='best')
plt.title(
    f"클릭 기반 초기 타원 | major={major_ax:.1f}px, "
    f"minor={minor_ax:.1f}px, PA={position_angle_deg:.1f}°"
)
plt.colorbar()
plt.show()

print("녹색 점선 타원이 맞지 않으면 위 클릭 셀을 다시 실행해 재선택하세요.")


In [ ]:
# Colab 클릭 방식은 별도 Matplotlib widget backend를 사용하지 않습니다.
print("✅ Colab 브라우저 클릭 캡처 방식 사용 중")


* 위에서 선택한 클릭 좌표를 이후 등광도 분석의 초기 타원값으로 사용합니다.


In [ ]:
# 클릭값 유효성 검사
if major_ax is None or minor_ax is None:
    raise RuntimeError("먼저 위 셀에서 장축·단축 끝점을 클릭하세요.")
if major_ax <= 0 or minor_ax <= 0:
    raise ValueError("장축/단축 반경 계산값이 올바르지 않습니다.")
if minor_ax > major_ax:
    raise ValueError("단축 반경이 장축보다 큽니다. 클릭 셀을 다시 실행하세요.")
print("✅ 클릭 기반 초기 타원 설정 완료")


* 아래 그림에서 각 필터의 녹색 점선 타원이 은하 형태와 대략 맞는지 확인합니다. 맞지 않으면 위 클릭 셀을 다시 실행해 두 점을 다시 선택하세요.


In [ ]:
# 클릭한 값으로 각 필터의 타원 기하학 생성 및 확인
vmax, vmin = 90., 15.
fig = plt.figure(figsize=(18,9))
n = 1

for f in filter:
    cx, cy = globals()['cen_{}'.format(f)]

    globals()['geometry_{}'.format(f)] = EllipseGeometry(
        x0=cx,
        y0=cy,
        sma=major_ax,
        eps=ellipse,
        pa=position_angle_rad,
        astep=0.1,
        linear_growth=True
    )

    globals()['aper_{}'.format(f)] = EllipticalAperture(
        (cx, cy),
        major_ax,
        minor_ax,
        position_angle_rad
    )

    maxv = np.percentile(globals()['smo_{}'.format(f)], vmax)
    minv = np.percentile(globals()['smo_{}'.format(f)], vmin)

    ax = fig.add_subplot(2,3,n)
    im = plt.imshow(
        globals()['smo_{}'.format(f)],
        cmap='gray_r',
        vmax=maxv,
        vmin=minv,
        origin='lower'
    )
    plt.contour(
        globals()['smo_{}'.format(f)],
        levels=globals()['levels_{}'.format(f)],
        origin='lower',
        colors='aqua',
        linewidths=0.8
    )
    globals()['aper_{}'.format(f)].plot(
        color='greenyellow',
        linewidth=2.5,
        linestyle='dashed'
    )
    plt.grid(color='white', ls='--', alpha=0.35)
    plt.colorbar(im)
    plt.title(
        f"{f}-band | major={major_ax:.1f}, minor={minor_ax:.1f}, "
        f"PA={position_angle_deg:.1f}°"
    )
    n += 1

plt.tight_layout()
plt.show()


In [ ]:
# 이미지 저장

save_file_name = str(ring_num)+'_2_ellipsegeometry_contour.jpg'
plt.savefig(save_path + save_file_name)

## **등광도 분석**

### **등광도 맞춤 분석 — Colab 기본값**


* 첫 실행에서는 아래 기본값을 권장합니다.
* 결과가 은하 영상과 잘 맞지 않을 때만 step, nclip, fflag 값을 조정해 다시 실행하세요.


In [ ]:
#@title ⑥ 등광도 맞춤 옵션
iso_step = 1.0 #@param {type:"number"}
iso_dist = 0.396 #@param {type:"number"}
iso_nclip = 0 #@param {type:"integer"}
iso_fflag = 0.7 #@param {type:"number"}

print("step:", iso_step)
print("arcsec/pixel:", iso_dist)
print("nclip:", iso_nclip)
print("fflag:", iso_fflag)

for f in filter:
    globals()['r_pix_{}'.format(f)] = []
    globals()['mu_{}'.format(f)] = []
    globals()['ep_{}'.format(f)] = []
    globals()['ep_err_{}'.format(f)] = []
    globals()['pa_{}'.format(f)] = []
    globals()['pa_err_{}'.format(f)] = []

for f in filter:
    try:
        ellipse_fit = Ellipse(globals()['smo_{}'.format(f)],
                              geometry=globals()['geometry_{}'.format(f)])
        isolist = ellipse_fit.fit_image(
            minsma=0,
            step=iso_step,
            linear=True,
            maxsma=major_ax,
            fix_center=True,
            nclip=iso_nclip,
            fflag=iso_fflag
        )
        globals()['ellipse_{}'.format(f)] = ellipse_fit
        globals()['isolist_{}'.format(f)] = isolist

        intens = -2.5*np.log10(isolist.intens)
        pa_deg = isolist.pa.copy()/np.pi*180.
        pa_deg[pa_deg < 90] = pa_deg[pa_deg < 90] + 180
        pa_deg = pa_deg - 90.
        pa_err_deg = isolist.pa_err/np.pi*180.

        globals()['intens_{}'.format(f)] = intens
        globals()['mu_{}'.format(f)].append(np.around(intens, 3))
        globals()['ep_{}'.format(f)].append(np.around(isolist.eps, 3))
        globals()['ep_err_{}'.format(f)].append(np.around(isolist.ellip_err, 3))
        globals()['pa_{}'.format(f)].append(np.around(pa_deg, 3))
        globals()['pa_err_{}'.format(f)].append(np.around(pa_err_deg, 3))

        globals()['r_pix_{}'.format(f)] = list(range(len(isolist.sma)))
        globals()['r_arc_{}'.format(f)] = np.around(
            np.asarray(globals()['r_pix_{}'.format(f)]) * iso_dist, 2
        )
        print(f"✅ {f}-band 등광도 맞춤 완료: {len(isolist.sma)} points")
    except Exception as e:
        print(f"⚠️ {f}-band 맞춤 실패:", e)


 ### **등광도 지도 작성(Isophotal contour map)**

* 각 필터별 등광도 지도를 통해 실제 등광도선과 분석된 등광도선이 일치하는지 확인합니다.
* 불일치가 심하다면 은하 내부에 천체가 존재하거나 내부가 복잡하여 이상값을 유발하였기 때문입니다.
* 은하 내부에 천체가 존재한다면, 천체마스킹 작업을 합니다.
* 은하 내부가 복잡하다면, 등광도 맞춤 분석에서 옵션을 조정합니다.
* 은하 내부에 천체가 있는 이유는, 사실 시선 방향에 있던 별과 겹쳐서 마치 은하 내부에 천체가 있는 것처럼 이미지에 투영되었기 때문입니다. 즉, 은하 내부에는 별이 없습니다.
* 은하 내부가 복잡한 이유는, 은하를 희미한 밝기를 더 많이 포착하기 위해 등광도선을 많이 그렸기 때문입니다. 

* 등광도 맞춤 분석 결과를 토대로 등광도 관련 밝기 강도, 타원율, 위치각 특징을 분석함으로써 은하의 특징을 파악합니다.

In [ ]:
# 분석된 등광도곡선 그리기

for f in filter:
    min_pix = 10 # 분석된 등광도 곡선 최소 픽셀 지점(시작점)
    globals()['max_pix_{}'.format(f)] = int(major_ax) # 등광도 곡선 최대 픽셀 지점(끝점) -> 위에서 입력한 장반경 값과 같음
    step = 10

vmax = 90.
vmin = 15.
print('max_percetile', vmax)
print('min_percetile:', vmin)            

# 그림
fig = plt.figure( figsize = (14,9) )
n = 1
for f in filter :
    globals()['max_value_{}_temp'.format(f)] = np.percentile(globals()['smo_{}'.format(f)], vmax)
    globals()['min_value_{}_temp'.format(f)] = np.percentile(globals()['smo_{}'.format(f)], vmin)   
    ax = fig.add_subplot(2,3,n)
    ct = plt.contour(globals()['smo_{}'.format(f)], levels = globals()['levels_{}'.format(f)] , origin = 'lower', colors= 'black', linewidths = 0.5)
    #plt.xlim(globals()['cen_{}'.format(f)][0]-0, globals()['cen_{}'.format(f)][0]+0) # 이미지 확대를 위해 중심 기준 x축 방향 +-50 처리함
    #plt.ylim(globals()['cen_{}'.format(f)][1]-0, globals()['cen_{}'.format(f)][1]+0) # 이미지 확대를 위해 중심 기준 y축 방향 +-50 처리함
    ax.grid(color = 'white', ls = '--')
    ax.set_title(f + '_isophotal contour map')
    smas = np.linspace(min_pix, globals()['max_pix_{}'.format(f)], step)
    try:
        for sma in smas:
            iso = globals()['isolist_{}'.format(f)].get_closest(sma)
            x, y, = iso.sampled_coordinates()
            ax.plot(x, y, ls = '--', color = 'red', lw = 0.5)
    except:
        pass
    n += 1
plt.show()

In [ ]:
# 이미지 저장

save_file_name = str(ring_num)+'_3_isophote_contour.jpg'
plt.savefig(save_path + save_file_name)

### **등광도 관련 광도 윤곽(Luminosity Profile), 타원율(Ellipticity), 위치각(Position Angle) 특징 분석**

* 각 필터별로 분석된 다양한 등광도 맞춤 분석의 물리값을 확인합니다.

In [ ]:
# 등광도 맞춤 분석 물리값 표

display('isolist_u_table', isolist_u.to_table())
display('isolist_g_table', isolist_i.to_table())
display('isolist_r_table', isolist_i.to_table())
display('isolist_i_table', isolist_i.to_table())
display('isolist_z_table', isolist_z.to_table())

In [ ]:
# 등광도 맞춤 분석 물리값 데이터 프레임 변환

for f in filter:
    try:
        columns = ['R"', 'Mu', 'Ellipticity', 'Ellip_error', 'PA', 'PA_error']
        #globals()['df_r_{}'.format(f)] = pd.DataFrame(globals()['sma_{}'.format(f)]).transpose()
        globals()['df_r_{}'.format(f)] = pd.DataFrame(globals()['r_arc_{}'.format(f)]) # 각 필터별 등광도 분석된 장반경
        globals()['df_mu_{}'.format(f)] = pd.DataFrame(globals()['mu_{}'.format(f)]).transpose() # 각 필터별 등광도 분석된 광도 값
        globals()['df_ep_{}'.format(f)] = pd.DataFrame(globals()['ep_{}'.format(f)]).transpose() # 각 필터별 등광도 분석된 타원율 값
        globals()['df_ep_err_{}'.format(f)] = pd.DataFrame(globals()['ep_err_{}'.format(f)]).transpose() # 각 필터별 등광도 분석된 타원율 에러 값
        globals()['df_pa_{}'.format(f)] = pd.DataFrame(globals()['pa_{}'.format(f)]).transpose() # 각 필터별 등광도 분석된 위치각 값
        globals()['df_pa_err_{}'.format(f)] = pd.DataFrame(globals()['pa_err_{}'.format(f)]).transpose() # 각 필터별 등광도 분석된 위치각 에러 값
    
        globals()['df_my_eps_{}'.format(f)] = pd.concat([globals()['df_r_{}'.format(f)], globals()['df_mu_{}'.format(f)], globals()['df_ep_{}'.format(f)], globals()['df_ep_err_{}'.format(f)], globals()['df_pa_{}'.format(f)], globals()['df_pa_err_{}'.format(f)]], axis = 1)
        globals()['df_my_eps_{}'.format(f)].columns = columns
    except:
        pass

In [ ]:
# csv 파일 저장

df_all = []
for f in filter:
    cc = globals()['df_my_eps_{}'.format(f)].rename(columns = lambda x: f +'_' + x)
    df_all.append(cc)
    save_file_name1 = str(ring_num)+'_4_'+ f + '_eps__result.csv'
    globals()['df_my_eps_{}'.format(f)].to_csv(save_path + save_file_name1, header=True, index=False)

save_file_name2 = str(ring_num)+'_eps__result.csv' 
df_all_concat = pd.concat(df_all, axis=1)
df_all_concat.to_csv(save_path + save_file_name2, header=True, index=False)   
display(df_all_concat)

* 에러 값을 포함한 광도 윤곽의 광도와 타원율, 위치각의 변화를 분석함으로써 은하의 특징을 살펴봅니다.

In [ ]:
# 그래프 시각화 에러 값 포함

plt.ion()
plt.figure(figsize=(6, 3.5), dpi=300)
for i,j,k,f in zip(range(1,17,3), range(2,18,3), range(3,19,3), filter):
    plt.subplot(5,3,i)
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Mu'][10:], 'r^-', ms=0.5, linewidth=0.2, label= f)
    plt.xlabel('Major_R(arcsec)', fontsize = 3 ,labelpad = 1) 
    plt.ylabel('Magnitude', fontsize = 3, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.gca().invert_yaxis()
    plt.legend(loc = 'best', fontsize = 3.5)

    plt.subplot(5,3,j)
    plt.errorbar(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Ellipticity'][10:], globals()['df_my_eps_{}'.format(f)]['Ellip_error'][10:], fmt='r^-', ms=0.5, linewidth=0.2, elinewidth=0.2, ecolor='black', capsize=1, capthick=0.5, label= f)
    plt.xlabel('Major_R(arcsec)', fontsize = 3 ,labelpad = 1) 
    plt.ylabel('Ellipticity', fontsize = 3, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.legend(loc = 'best', fontsize = 3.5)

    plt.subplot(5,3,k)
    plt.errorbar(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['PA'][10:], globals()['df_my_eps_{}'.format(f)]['PA_error'][10:], fmt='r^-', ms=0.5, linewidth=0.2, elinewidth=0.2, ecolor='black', capsize=1, capthick=0.5, label= f)
    plt.xlabel('Major_R(arcsec)', fontsize = 3 ,labelpad = 1)
    plt.ylabel('Position Angle (deg)', fontsize = 3, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.legend(loc = 'best', fontsize = 3.5)

plt.show()

In [ ]:
# 이미지 저장

save_file_name = str(ring_num)+'_4_result_graph(with error).jpg'
plt.savefig(save_path + save_file_name)

* 에러 값을 미포함한 광도 윤곽의 광도와 타원율, 위치각의 변화를 분석함으로써 은하의 특징을 살펴봅니다.

In [ ]:
# 그래프 시각화 에러 값 미포함

plt.ion()
plt.figure(figsize=(6, 3.5), dpi=300)
for i,j,k,f in zip(range(1,17,3), range(2,18,3), range(3,19,3), filter):
    plt.subplot(5,3,i)
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Mu'][10:], 'r^-', ms=0.5, linewidth=0.2, label= f)
    plt.xlabel('Major_R(arcsec)', fontsize = 3 ,labelpad = 1) 
    plt.ylabel('Magnitude', fontsize = 3, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.gca().invert_yaxis()
    plt.legend(loc = 'best', fontsize = 3.5)

    plt.subplot(5,3,j)
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Ellipticity'][10:], 'r^-', ms=0.5, linewidth=0.2, label= f)
    plt.xlabel('Major_R(arcsec)', fontsize = 3 ,labelpad = 1) 
    plt.ylabel('Ellipticity', fontsize = 3, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.legend(loc = 'best', fontsize = 3.5)

    plt.subplot(5,3,k)
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['PA'][10:], 'r^-', ms=0.5, linewidth=0.2, label= f)
    plt.xlabel('Major_R(arcsec)', fontsize = 3 ,labelpad = 1)
    plt.ylabel('Position Angle (deg)', fontsize = 3, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 1.5, width = 0.5, labelsize = 3, pad = 1)
    plt.legend(loc = 'best', fontsize = 3.5)

plt.show()

In [ ]:
# 이미지 저장

save_file_name = str(ring_num)+'_5_result_graph(without error).jpg'
plt.savefig(save_path + save_file_name)

### **랜덤포레스트 회귀분석을 통한 등광도 특징 보조 분석**

* 랜덤포레스트 회귀분석을 통한 등광도 특징 보조 분석은 실제 등광도선과 분석된 등광도선의 불일치가 심할때 합니다.
* 이 분석은 등광도 맞춤 분석의 옵션 조정 후 이상값 회피를 통한 회귀분석으로, 더 정확한 등광도 관련 광도 윤곽, 타원율, 위치각을 도출해 냅니다.  

In [ ]:
#@title ⑦ Random Forest 보조 분석 필터 선택
sel = "r" #@param ["u", "g", "r", "i", "z"]

r = globals()['df_my_eps_{}'.format(sel)][['R"']]
mu = globals()['df_my_eps_{}'.format(sel)].Mu
ep = globals()['df_my_eps_{}'.format(sel)].Ellipticity
pa = globals()['df_my_eps_{}'.format(sel)].PA

print("선택 필터:", sel)


In [ ]:
# 머신러닝 랜덤포레스트 회귀

# 랜덤포레스트 초매개변수 설정
rfr = RandomForestRegressor()

param_grid = {'n_estimators': [1000], 'max_depth' : [4], 'min_samples_leaf' : [5], 'bootstrap': [True]}  # 초매개변수 설정
grid_search = GridSearchCV(rfr, param_grid, scoring='neg_mean_squared_error', return_train_score=True) # 성능 측정 지표 설정

# 랜덤포레스트 훈련 및 예측
grid_search.fit(r, mu) # 밝기 강도 예측
y0_pred = grid_search.predict(r)

grid_search.fit(r, ep) # 타원율 예측
y1_pred = grid_search.predict(r)

grid_search.fit(r, pa) # 위치각 예측
y2_pred = grid_search.predict(r)

In [ ]:
# 머신러닝 반영 회귀 그래프

plt.ion()
plt.figure(figsize=(4,4.5), dpi=300)
for f in sel:
    plt.subplot(3,1,1)
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Mu'][10:], 'r^-', ms=7, mfc='none', linewidth=1, alpha = 0.5, label= f + '_Ori')
    plt.plot(r[10:], y0_pred[10:], 'gx:', ms=7, mec='green', mew=1, mfc='none', linewidth=1.5, alpha = 1, label= f + '_RFR')
    plt.ylabel('Magnitude', fontsize = 9.5, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 3.5, width = 2, labelsize = 9.5, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 3.5, width = 2, labelsize = 9.5, pad = 1)
    plt.gca().invert_yaxis()
    plt.legend(loc = 'best', fontsize = 9.5)

    plt.subplot(3,1,2)
    #plt.errorbar(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Ellipticity'][10:], globals()['df_my_eps_{}'.format(f)]['Ellip_error'][10:], fmt='r^-', ms=7, mfc='none', linewidth=1, elinewidth=1, ecolor='black', capsize=2, capthick=2.5, alpha = 0.5, label= f + '_Ori')
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['Ellipticity'][10:], 'r^-', ms=7, mfc='none', linewidth=1, alpha = 0.5, label= f + '_Ori')
    plt.plot(r[10:], y1_pred[10:], 'gx:', ms=7, mec='green', mew=1, mfc='none', linewidth=1.5, alpha = 1, label= f + '_RFR')

    plt.ylabel('Ellipticity', fontsize = 9.5, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 3.5, width = 2, labelsize = 9.5, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 3.5, width = 2, labelsize = 9.5, pad = 1)
    plt.legend(loc = 'upper right', fontsize = 9.5)

    plt.subplot(3,1,3)
    #plt.errorbar(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['PA'][10:], globals()['df_my_eps_{}'.format(f)]['PA_error'][10:], fmt='r^-', ms=7, mfc='none', linewidth=1, elinewidth=1, ecolor='black', capsize=2, capthick=2.5, alpha = 0.5, label= f + '_Ori')
    plt.plot(globals()['df_my_eps_{}'.format(f)]['R"'][10:], globals()['df_my_eps_{}'.format(f)]['PA'][10:], 'r^-', ms=7, mfc='none', linewidth=1, alpha = 0.5, label= f + '_Ori')
    plt.plot(r[10:], y2_pred[10:], 'gx:', ms=7, mec='green', mew=1, mfc='none', linewidth=1.5, alpha = 1, label= f + '_RFR')
    plt.xlabel('Major_R(arcsec)', fontsize = 9.5,labelpad = 1)
    plt.ylabel('Position Angle (deg)', fontsize = 9.5, labelpad = 1)
    plt.xticks()
    plt.yticks()
    plt.tick_params(axis = 'x', direction = 'out', length = 3.5, width = 2, labelsize = 9.5, pad = 1)
    plt.tick_params(axis = 'y', direction = 'out', length = 3.5, width = 2, labelsize = 9.5, pad = 1)
    plt.legend(loc = 'right', fontsize = 9.5)

    plt.show()

In [ ]:
# 이미지 저장

save_file_name = str(ring_num)+'_6_regression_result_graph.jpg'
plt.savefig(save_path + save_file_name)

## 📦 결과 파일 내려받기

분석 과정에서 생성된 CSV와 JPG 결과는 `/content/ring_galaxy/result/`에 저장됩니다.  
아래 셀을 실행하면 결과 폴더를 ZIP으로 묶어 다운로드할 수 있습니다.


In [ ]:
# 결과 폴더 ZIP 압축 및 다운로드
import shutil
from google.colab import files

zip_file = f"/content/outer_ring_{ring_num}_results"
shutil.make_archive(zip_file, 'zip', result_dir)
print("✅ 결과 ZIP 생성:", zip_file + ".zip")
files.download(zip_file + ".zip")
